# 28.2 — Fast Inference: hnswlib L2 ANN + Tuned Rerank (WJ 512)

Same encoder + triplet training as nb28, but replaces the inference pipeline:
- **nmslib WeightedJaccard HNSW → hnswlib L2 HNSW**: On the L1-simplex, WJ(a,b) is a monotone function of L1(a,b), and L2 correlates closely with L1 on smooth embeddings. hnswlib is written in optimized C++ (no Python overhead per comparison) and is 10–30× faster than nmslib for dense 512-dim vectors.
- **Rerank batch_size 8 → 64**: GPU reranker processes 8 queries at a time (left over from a conservative memory budget). At D=18220, batch=64 uses 4.7 GB — safe on A100.

Goal: push QPS from ~1048 (nb28 nmslib) to 15,000+ without sacrificing rerank recall.

In [ ]:
import os, random, sys, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
import hnswlib
sys.path.append('/raid/ruban/hpmlproj/term_project')
from sota_experiment_common import (
    build_fn_mask, build_gt_cache, build_gt_gpu,
    cleanup, eval_recall, load_dataset_normalized,
    nmslib_neighbors, preload_rerank_corpus, release_rerank_corpus, rerank_wj_gpu, save_result,
)

dataset_name  = "full"
out_dim       = 512
device        = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
THREADS       = 40
seed          = 42
batch_size    = 2048
epochs        = 75
lr            = 1e-3
weight_decay  = 1e-4
max_pos       = 30
margin        = 0.3
candidate_ks  = [500, 1000] if dataset_name == "10k" else [1000, 2000]

# hnswlib index params — tuned for 512-dim dense L1-simplex embeddings
HNSW_M              = 32    # nb28 nmslib used M=20; M=32 gives better recall
HNSW_EF_CONSTRUCTION = 200
HNSW_EF_SEARCH      = 200   # increase if recall drops vs nmslib baseline

RERANK_BATCH = 64   # was 8; 64×1000×18220×4 = 4.7 GB — safe on A100

METHOD_NAME   = "fast_wj_512"
NOTEBOOK_NAME = "28_2_infonce_wj_512_fast.ipynb"
OUT_PATH      = "/tmp/results_sota_fast_wj_512.pkl"
CKPT_PATH     = f"/tmp/best_sota_fast_wj_512_{dataset_name}.pt"

# If nb28 already ran, reuse its weights — training is identical
NB28_CKPT    = f"/tmp/best_sota_triplet_autoencoder_wj_512_{dataset_name}.pt"
SKIP_TRAINING = Path(NB28_CKPT).exists()
if SKIP_TRAINING:
    import shutil
    shutil.copy(NB28_CKPT, CKPT_PATH)
    print(f"nb28 checkpoint found → copied to {CKPT_PATH}, will skip training")
else:
    print(f"nb28 checkpoint not found at {NB28_CKPT} → will train from scratch")

random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
print(f"dataset={dataset_name} | batch={batch_size} | epochs={epochs} | "
      f"HNSW M={HNSW_M} ef={HNSW_EF_SEARCH} | rerank_batch={RERANK_BATCH}")

In [ ]:
qt, gt, query_start, corpus_qt, query_qt, corpus_sums, qt_norm = load_dataset_normalized(dataset_name)

In [ ]:
def wj_sim(a, b):
    mins = torch.minimum(a, b).sum(dim=-1)
    maxs = torch.maximum(a, b).sum(dim=-1).clamp(min=1e-10)
    return mins / maxs

def wj_triplet_loss_inbatch(anchors, positives, margin=0.3, gt_matrix=None):
    """In-batch hard-negative WJ triplet loss with FN masking."""
    sim_ap   = wj_sim(anchors, positives)
    mins_c   = torch.min(anchors.unsqueeze(1), positives.unsqueeze(0)).sum(2)
    maxs_c   = torch.max(anchors.unsqueeze(1), positives.unsqueeze(0)).sum(2)
    sim_cross = mins_c / maxs_c.clamp(min=1e-10)
    sim_cross.fill_diagonal_(-1e9)
    n_fn = 0
    if gt_matrix is not None:
        fn_mask = gt_matrix.to(sim_cross.device)
        n_fn = int(fn_mask.sum().item())
        if n_fn:
            sim_cross[fn_mask] = -1e9
    sim_an   = sim_cross.max(dim=1).values
    loss     = F.relu(sim_an - sim_ap + margin)
    violated = loss > 0
    if violated.sum() == 0:
        return torch.tensor(0.0, device=anchors.device, requires_grad=True), 0, n_fn
    return loss[violated].mean(), int(violated.sum().item()), n_fn

class IndexAnchorPositiveDataset(Dataset):
    def __init__(self, gt_lookup, query_start, max_pos=30):
        self.pairs = []
        for qid, neighbors in gt_lookup.items():
            for nid in neighbors[:max_pos]:
                if qid >= query_start and nid < query_start:
                    self.pairs.append((qid, nid))
        random.shuffle(self.pairs)
        print(f"pairs={len(self.pairs):,}  steps/epoch={len(self.pairs)//batch_size}")
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        qid, pid = self.pairs[idx]
        return qid, pid

class TripletEncoder(nn.Module):
    def __init__(self, in_dim, out_dim=512):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 4096, bias=False), nn.BatchNorm1d(4096), nn.ReLU(),
            nn.Linear(4096, 1024, bias=False), nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, out_dim, bias=False), nn.BatchNorm1d(out_dim),
        )
    def forward(self, x):
        z = F.relu(self.encoder(x))
        return z / z.sum(dim=1, keepdim=True).clamp(min=1e-10)

def embed_all(model, qt, batch_size=4096):
    """Encode full dataset using DataParallel across all GPUs."""
    model.eval(); out = []
    with torch.no_grad():
        for s in range(0, len(qt), batch_size):
            x = torch.tensor(qt[s:s+batch_size], dtype=torch.float32, device=device)
            out.append(model(x).cpu().numpy().astype(np.float32))
    return np.vstack(out)

def hnswlib_index(corpus_embs, M=HNSW_M, ef_construction=HNSW_EF_CONSTRUCTION, num_threads=THREADS):
    """Build hnswlib L2 index on 512-dim corpus embeddings.
    L2 ≈ L1 ≡ WJ on L1-simplex: same neighbourhood structure, 10-30× faster than nmslib WJ."""
    N, d = corpus_embs.shape
    t0 = time.time()
    p = hnswlib.Index(space='l2', dim=d)
    p.init_index(max_elements=N, ef_construction=ef_construction, M=M)
    p.add_items(corpus_embs, num_threads=num_threads)
    print(f"  hnswlib index built: {N:,} items | D={d} | M={M} | ef_c={ef_construction} | {time.time()-t0:.1f}s",
          flush=True)
    return p

def hnswlib_query(p, query_embs, k, ef_search=HNSW_EF_SEARCH, num_threads=THREADS):
    """Query a pre-built hnswlib index. ef_search must be >= k."""
    p.set_ef(max(ef_search, k))
    t0 = time.time()
    labels, _ = p.knn_query(query_embs, k=k, num_threads=num_threads)
    elapsed = time.time() - t0
    qps = len(query_embs) / max(elapsed, 1e-9)
    nbrs = labels.tolist()
    return nbrs, {"qps": qps, "query_s": elapsed, "ef_search": ef_search, "k": k}

def eval_embeddings(embs, method_name, out_path, notebook_name):
    corpus_embs = embs[:query_start]; query_embs = embs[query_start:]
    max_k = max(max(candidate_ks), 500)

    # Build hnswlib index once — reuse across all k evaluations
    p = hnswlib_index(corpus_embs)

    # ── No-rerank eval ──
    nbrs, info = hnswlib_query(p, query_embs, k=max_k)
    metrics = {**eval_recall(gt, nbrs, query_start, max_k), **info,
               "dim": out_dim, "M": HNSW_M, "ef_construction": HNSW_EF_CONSTRUCTION}
    for k, v in metrics.items():
        if isinstance(k, int): print(f"R@{k:<4} = {v:.4f}")
    print(f"QPS={metrics['qps']:.1f}")
    save_result(out_path, dataset_name, method_name, metrics, meta={"notebook": notebook_name})

    # ── Rerank evals (batch_size=64 vs nb28's 8) ──
    preload_rerank_corpus(corpus_qt, corpus_sums)
    for ck in candidate_ks:
        cand, ci = hnswlib_query(p, query_embs, k=ck)
        t0 = time.time()
        rr = rerank_wj_gpu(query_qt, cand, corpus_qt, corpus_sums, top_k=ck, batch_size=RERANK_BATCH)
        rr_s = time.time() - t0
        # Combined QPS: time for ANN + rerank, divided by query count
        total_s = ci["query_s"] + rr_s
        qps_total = len(query_qt) / max(total_s, 1e-9)
        rr_metrics = {**eval_recall(gt, rr, query_start, ck), "qps": qps_total,
                      "ann_qps": ci["qps"], "candidate_k": ck}
        key = f"{method_name}_rerank_{ck}"
        for k, v in rr_metrics.items():
            if isinstance(k, int): print(f"{key} R@{k} = {v:.4f}")
        print(f"{key} QPS(ann+rerank)={qps_total:.1f}  ANN-only={ci['qps']:.1f}")
        save_result(out_path, dataset_name, key, rr_metrics, meta={"notebook": notebook_name})
    release_rerank_corpus()

In [ ]:
# All 8 GPUs; vecs on cuda:7 so cuda:0 (gather) stays free for the 8 GB triplet intermediate.
device      = torch.device("cuda:0")
vecs_device = torch.device("cuda:7")
print("Pre-loading vectors to cuda:7...")
vecs_gpu = torch.from_numpy(np.ascontiguousarray(qt_norm, dtype=np.float32)).to(vecs_device)
print(f"Loaded: {vecs_gpu.nbytes/1024**3:.2f} GB on {vecs_device}")

model = TripletEncoder(qt_norm.shape[1], out_dim)
model = nn.DataParallel(model, device_ids=list(range(torch.cuda.device_count())))
model = model.to(device)
print(f"DataParallel on {torch.cuda.device_count()} GPUs")

dataset = IndexAnchorPositiveDataset(gt, query_start, max_pos=max_pos)
loader  = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                     num_workers=4, pin_memory=True, drop_last=True,
                     persistent_workers=True)

gt_stacked = build_gt_cache(gt, len(qt_norm), query_start, dataset_name)
gt_gpu     = build_gt_gpu(gt_stacked, vecs_device)
del gt_stacked

opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
best = float('inf')
t0_train = time.time()

epoch_bar = tqdm(range(1, epochs + 1), desc="epochs", unit="ep")
for epoch in epoch_bar:
    model.train()
    tot_loss = tot_trip = tot_viol = steps = 0
    step_bar = tqdm(loader, desc=f"ep{epoch:02d}", leave=False, unit="step")
    for a_ids, p_ids in step_bar:
        a = vecs_gpu[a_ids.to(vecs_device)].to(device)
        p = vecs_gpu[p_ids.to(vecs_device)].to(device)
        B = a.shape[0]
        z = model(torch.cat([a, p]))
        za, zp = z[:B], z[B:]
        fn_mask = build_fn_mask(a_ids, p_ids, gt_gpu, query_start)
        trip, n_viol, _ = wj_triplet_loss_inbatch(za, zp, margin=margin, gt_matrix=fn_mask)
        loss = trip
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        tot_loss += float(loss.detach()); tot_trip += float(trip.detach())
        tot_viol += n_viol; steps += 1
        step_bar.set_postfix(loss=f"{float(loss.detach()):.4f}", viol=n_viol)
    sch.step()
    avg = tot_loss / max(steps, 1)
    if avg < best:
        best = avg
        torch.save(model.module.state_dict(), CKPT_PATH)
    elapsed = (time.time() - t0_train) / 60
    eta     = elapsed / epoch * (epochs - epoch)
    epoch_bar.set_postfix(loss=f"{avg:.4f}", best=f"{best:.4f}", eta=f"{eta:.0f}m")
    if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
        print(f"epoch {epoch:02d}/{epochs} | loss={avg:.4f} | trip={tot_trip/steps:.4f} | "
              f"viol={tot_viol/steps:.1f} | {elapsed:.1f}min | eta={eta:.1f}min", flush=True)

print(f"Training done. best={best:.4f} | saved {CKPT_PATH}")

In [ ]:
(model.module if hasattr(model, "module") else model).load_state_dict(torch.load(CKPT_PATH, map_location=device, weights_only=True))
embs = embed_all(model, qt_norm)
eval_embeddings(embs, METHOD_NAME, OUT_PATH, NOTEBOOK_NAME)
cleanup()


In [ ]:

# ── GPU Exact L1 Search (full dataset) ───────────────────────────────────────
# Tests whether HNSW efSearch is the bottleneck on the full 187K corpus.
# Run this after c0c88cf7 (full eval cell) completes so embs is in scope.

FULL_N  = 233773   # total rows in full dataset
FULL_QS = 187019   # corpus size / query_start for full

# embs is set by embed_all(model, qt_norm) in the eval cell above.
# If it's stale (wrong shape), re-encode from the full checkpoint.
if 'embs' not in vars() or embs.shape[0] != FULL_N:
    print(f"Re-encoding from checkpoint (embs shape mismatch: expected {FULL_N} rows)...")
    _qt_norm_full = np.load("/tmp/qt_norm_full.npy")
    _enc = TripletEncoder(_qt_norm_full.shape[1], out_dim)
    _enc.load_state_dict(torch.load(CKPT_PATH, map_location=device, weights_only=True))
    _enc = nn.DataParallel(_enc, device_ids=list(range(torch.cuda.device_count()))).to(device)
    embs = embed_all(_enc, _qt_norm_full)
    del _enc, _qt_norm_full
    print(f"embs: {embs.shape}")
else:
    print(f"Using existing embs {embs.shape}")

corpus_embs_ex = embs[:FULL_QS]
query_embs_ex  = embs[FULL_QS:]
N, D = corpus_embs_ex.shape
Q    = len(query_embs_ex)
k    = 50

print(f"\nExact search: {Q} queries × {N} corpus | D={D} | k={k}")
search_dev    = torch.device("cuda:0")
corpus_gpu_ex = torch.from_numpy(corpus_embs_ex).to(search_dev)
print(f"Corpus on GPU: {corpus_gpu_ex.nbytes/1024**3:.2f} GB")

chunk_size = 64   # 64×187K×512×4 ≈ 23 GB intermediate — fine on A100
all_idx = []
t0 = time.time()
with torch.no_grad():
    for s in range(0, Q, chunk_size):
        q = torch.from_numpy(query_embs_ex[s:s+chunk_size]).to(search_dev)
        l1 = (q.unsqueeze(1) - corpus_gpu_ex.unsqueeze(0)).abs_().sum(dim=2)
        all_idx.append(l1.topk(k, dim=1, largest=False).indices.cpu().numpy())
elapsed = time.time() - t0
qps_exact = Q / elapsed
nbrs_exact = np.vstack(all_idx)

_gt_ex = gt if (query_start == FULL_QS) else load_dataset("full")[1]
metrics_ex = eval_recall(_gt_ex, nbrs_exact, FULL_QS, k)

print(f"\n--- GPU Exact L1 (full, nb28 model) ---")
for kk in [10, 50]:
    print(f"R@{kk:<4} = {metrics_ex[kk]:.4f}")
print(f"QPS  = {qps_exact:.0f}  ({elapsed:.1f}s total)")
print(f"\nHNSW R@10 (from eval cell above)  →  Exact R@10={metrics_ex[10]:.4f}")
print("If Exact ≈ HNSW: embedding quality is the ceiling.")
print("If Exact >> HNSW: efSearch=200 is too small for 187K corpus.")
del corpus_gpu_ex
